# Employee Dataset Cleaning — Business_Operation.csv
### Employee Management Module — AI Business Operations Management Platform

**Prepared for:** First Scrum Review (Ticket 1 — Data & System)
**Prepared by:** Akhilesh P. S


**Allocation logic this dataset supports:**
`Required Skill → Skill Match → Availability → Workload → Recommended Employee`


## 1. Setup

Import pandas for data handling. 

In [ ]:
import pandas as pd

RAW_PATH = "Business_Operation.csv"
ML_READY_PATH = "Business_Operation_ml_ready.csv"

PROTECTED_COLUMNS = [
    "employee_id",
    "skills",
    "required_skill",
    "availability_status",
    "workload_percentage",
]

## 2. Load the raw data

We load the raw CSV and immediately check that the 5 protected columns exist.
**The raw file is never modified — we only ever read from it.** All cleaned
output goes to separate files, so the original data stays available for
anyone to verify our work against.

In [15]:
df_raw = pd.read_csv("../raw/Business_Operation.csv")

missing = [c for c in PROTECTED_COLUMNS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Protected columns missing from raw file: {missing}")

print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (50250, 25)


,employee_id,employee_name,email,department,job_role,experience_years,hire_date,skills,required_skill,availability_status,...,task_description,task_priority,task_status,task_start_date,task_deadline,estimated_hours,hours_logged,progress_percentage,allocation_score,recommended_employee
0,EMP2373,Pooja Menon,employee2373@company.local,Engineering,Developer,4.1,2013-09-01,NaN,Computer Vision,Busy,...,Complete Computer Vision related work for the ...,High,In Progress,2026-02-22,2026-02-25,36.5,25.1,76.8,16.5,1
1,EMP4021,Priya Menon,employee4021@company.local,HR,Analyst,6.1,2024-05-28,"SQL, Project Management, Computer Vision, Excel",NLP,Available,...,Complete NLP related work for the assigned bus...,Medium,In Progress,2025-11-13,2025-11-23,44.4,21.9,56.9,43.8,1
2,EMP2457,Swati Kapoor,employee2457@company.local,Engineering,Project Manager,2.1,2018-10-19,Computer Vision,Project Management,Available,...,Complete Project Management related work for t...,Medium,In Progress,2025-02-22,2025-03-31,32.0,21.2,56.6,41.9,1
3,EMP2344,Vivek Singh,employee2344@company.local,Finance,Data Scientist,4.5,2022-12-14,"SQL, Excel, Deep Learning, Computer Vision",Machine Learning,On Leave,...,Complete Machine Learning related work for the...,Medium,Completed,2025-11-24,2025-12-13,43.1,40.4,96.7,22.5,0
4,EMP205,Anita Pillai,employee205@company.local,Operations,Senior Developer,13.6,2022-09-14,"Python, NLP, Project Management, Django",Finance,Available,...,Complete Finance related work for the assigned...,Medium,Blocked,2026-05-27,2026-06-17,33.5,29.4,92.8,49.2,1


In [16]:
df.info()

<class 'pandas.DataFrame'>
Index: 50000 entries, 0 to 50249
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           50000 non-null  str    
 1   employee_name         49500 non-null  str    
 2   email                 50000 non-null  str    
 3   department            50000 non-null  string 
 4   job_role              50000 non-null  string 
 5   experience_years      49000 non-null  float64
 6   hire_date             50000 non-null  str    
 7   skills                48750 non-null  str    
 8   required_skill        49101 non-null  str    
 9   availability_status   49250 non-null  string 
 10  workload_percentage   48913 non-null  float64
 11  active_tasks          50000 non-null  int64  
 12  performance_score     48750 non-null  float64
 13  task_id               50000 non-null  str    
 14  task_title            49500 non-null  str    
 15  task_description      49400 non-nul

## 3. Initial data quality check

Checked for missing values and duplicates


In [17]:
print("Missing values per column:")
print(df_raw.isnull().sum())
print("\nDuplicate rows:", df_raw.duplicated().sum())

Missing values per column:
employee_id                0
employee_name            502
email                      0
department               754
job_role                 904
experience_years        1004
hire_date                  0
skills                  1255
required_skill           903
availability_status      757
workload_percentage     1091
active_tasks               0
performance_score       1262
task_id                    0
task_title               504
task_description         604
task_priority            754
task_status              605
task_start_date            0
task_deadline              0
estimated_hours          903
hours_logged             995
progress_percentage     1003
allocation_score           0
recommended_employee       0
dtype: int64

Duplicate rows: 250


## 4. Remove exact duplicate rows

250 rows are byte-for-byte identical to another row. These add no new
information and would bias any model trained on this data 

In [18]:
df = df_raw.copy()
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")
print("Shape after dedup:", df.shape)

Dropped 250 duplicate rows
Shape after dedup: (50000, 25)


## 5. Normalize inconsistent text casing and spacing

fixing values like `'Marketing'`, `' marketing '`, and `'MARKETING'` 


In [19]:
def normalize_text_column(series):
    return series.astype("string").str.strip().str.title()

for col in ["department", "job_role", "availability_status",
            "task_priority", "task_status"]:
    df[col] = normalize_text_column(df[col])

df[["department", "job_role"]].head()

,department,job_role
0,Engineering,Developer
1,Hr,Analyst
2,Engineering,Project Manager
3,Finance,Data Scientist
4,Operations,Senior Developer


## 6. Fixing abbreviations and acronym 

Two problems found by manually inspecting unique values after step 5:

1. `"Engg"` is used as shorthand for `"Engineering"` in some rows — same
   real value, different spelling, not just a casing issue.
2. Title Case broke acronyms: `"HR"` became `"Hr"` and `"IT"` became `"It"`.
   Title Case assumes normal words, not acronyms, so this needed a manual fix.



In [20]:
dept_alias = {"Engg": "Engineering", "Hr": "HR", "It": "IT"}
df["department"] = df["department"].replace(dept_alias)
df["job_role"] = df["job_role"].str.replace(r"\bHr\b", "HR", regex=True)

sorted(df["department"].dropna().unique())

['Data Science',
 'Engineering',
 'Finance',
 'HR',
 'IT',
 'Marketing',
 'Operations',
 'Sales']

## 7. filled missing employee attributes from their own records

`department` and `job_role` describe the **employee**, not the task — so if
one row for an employee is missing `department` but another row for the
*same* `employee_id` has it, we should use that value rather than guessing
or leaving it blank. This is more accurate than filling with a generic
placeholder, because the real answer is already sitting elsewhere in the data.

Only if an employee has **no** valid value anywhere do we fall back to
`"Unknown"` (handled in the next step).

In [ ]:
for col in ["department", "job_role"]:
    df[col] = df.groupby("employee_id")[col].transform(
        lambda s: s.fillna(s.mode().iat[0]) if s.mode().size else s
    )


conflicts = df.groupby("employee_id")["department"].nunique()
print("Employees with >1 department value after backfill:", (conflicts > 1).sum())

Employees with >1 department value after backfill: 0


## 8. Cap out-of-range workload values

`workload_percentage` should logically max out at 100 (full capacity), but
500 rows exceed that. Rather than deleting these rows (losing real data),
we cap the value at 100 — this keeps the row usable while removing an
impossible value.

In [22]:
over_cap = (df["workload_percentage"] > 100).sum()
print(f"{over_cap} rows had workload_percentage > 100")

df["workload_percentage"] = df["workload_percentage"].clip(upper=100)
print("New max:", df["workload_percentage"].max())

500 rows had workload_percentage > 100
New max: 100.0


## 9. Handle remaining missing values — different strategy for each column type

Not all missing data should be treated the same way. three different
strategies depending on what kind of column it is:

| Column type | Strategy |
|---|---|
| Categorical/identity (`availability_status`, `task_priority`, `task_status`, `required_skill`) | Fill with `"Unknown"` |
| Free text (`employee_name`, `task_title`, `task_description`, `skills`) | Fill with `"Not specified"` |
| Numeric (`experience_years`, `workload_percentage`, `performance_score`, `estimated_hours`, `hours_logged`, `progress_percentage`) | Fill with the column **median** |

In [24]:

for col in ["department", "job_role"]:
    df[col] = df[col].fillna("Unknown")
for col in ["availability_status", "task_priority", "task_status", "required_skill"]:
    df[col] = df[col].fillna("Unknown")


for col in ["employee_name", "task_title", "task_description", "skills"]:
    df[col] = df[col].fillna("Not specified")


for col in ["experience_years", "workload_percentage", "performance_score",
            "estimated_hours", "hours_logged", "progress_percentage"]:
    df[col] = df[col].fillna(df[col].median())

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


## 10. Final integrity check

checked if any employees have different departments , also if the protected columns are all present

In [25]:
real_dept = df[df["department"] != "Unknown"]
dept_conflicts = real_dept.groupby("employee_id")["department"].nunique()
print("Genuine department conflicts:", (dept_conflicts > 1).sum())

missing_protected = [c for c in PROTECTED_COLUMNS if c not in df.columns]
print("Protected columns missing:", missing_protected if missing_protected else "None — all present")

print("\nFinal cleaned shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

Genuine department conflicts: 0
Protected columns missing: None — all present

Final cleaned shape: (50000, 25)
Duplicate rows: 0
Missing values: 0


## 11. Build the ML-ready version

The cleaned data so far is good conceptually, but a machine learning model
needs **numeric** input — it can't read text categories or comma-packed
skill lists directly. This section transforms the cleaned data into the
final model-ready format.

 Split the `skills` column and create a skill-match flag

`skills` currently holds multiple values in one cell (e.g.
`"SQL, Excel, Deep Learning"`). We split this into an actual list, then
create `skill_match` — a 1/0 flag for whether the employee's skills include
the task's `required_skill`. This is the single most direct feature for the
"Skill Match" step in the allocation logic.

In [28]:
df["skills_list"] = df["skills"].apply(
    lambda s: [] if s == "Not specified" else [x.strip() for x in s.split(",")]
)

df["skill_match"] = df.apply(
    lambda r: 1 if r["required_skill"] in r["skills_list"] else 0, axis=1
)

df[["skills", "required_skill", "skill_match"]].head(10)

,skills,required_skill,skill_match
0,Not specified,Computer Vision,0
1,"SQL, Project Management, Computer Vision, Excel",NLP,0
2,Computer Vision,Project Management,0
3,"SQL, Excel, Deep Learning, Computer Vision",Machine Learning,0
4,"Python, NLP, Project Management, Django",Finance,0
5,Project Management,Machine Learning,0
6,"Deep Learning, Excel, Data Analysis, Computer ...",Django,0
7,"Deep Learning, Computer Vision",SQL,0
8,React,Deep Learning,0
9,"React, Computer Vision, NLP, Machine Learning",Machine Learning,1


## 12. One-hot encode categorical columns

Models can't use text categories like `"Engineering"` or `"Available"`
directly — they need numbers. One-hot encoding turns each category into
its own 0/1 column (e.g. `department_Engineering`, `department_HR`).

We also one-hot encode individual skills (multi-label), since one employee
can have several skills at once.

In [29]:
all_skills = sorted({s for lst in df["skills_list"] for s in lst})
for skill in all_skills:
    col_name = "skill_" + skill.lower().replace(" ", "_")
    df[col_name] = df["skills_list"].apply(lambda lst, s=skill: 1 if s in lst else 0)

cat_cols = ["department", "job_role", "availability_status",
            "task_priority", "task_status", "required_skill"]
df_encoded = pd.get_dummies(df, columns=cat_cols, prefix=cat_cols)

print("Shape after encoding:", df_encoded.shape)

Shape after encoding: (50000, 77)


## 13. Drop columns that aren't useful as model inputs

- **Identifiers** (`task_id`) and **free text** (`employee_name`, `email`,
  `task_title`, `task_description`) carry no learnable pattern for this task.
- **Raw dates** (`hire_date`, `task_start_date`, `task_deadline`) aren't
  usable in their string form; `experience_years` already captures the
  useful signal from `hire_date`.
- **`skills`** (original text version) and **`skills_list`** are dropped
  since they've already been converted into the `skill_*` one-hot columns
  above.

We keep `employee_id` in this file for reference/joining, but it should
**not** be used as a model input — a model shouldn't learn from arbitrary
ID numbers.

In [31]:
drop_cols = ["employee_name", "email", "task_title", "task_description",
             "skills", "skills_list", "task_id", "hire_date",
             "task_start_date", "task_deadline"]
df_encoded = df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns])

df_encoded.to_csv("../cleaned/Business_Operation_ml_ready.csv", index=False)
print("Saved: ../cleaned/Business_Operation_ml_ready.csv")
print("Final ML-ready shape:", df_encoded.shape)

Saved: ../cleaned/Business_Operation_ml_ready.csv
Final ML-ready shape: (50000, 67)


## Summary

| Metric | Value |
|---|---|
| Raw rows | 50,250 |
| Duplicate rows removed | 250 |
| Rows after cleaning | 50,000 |
| Missing values remaining | 0 |
| Genuine department conflicts | 0 |
| Protected columns preserved | Yes, all 5 |
| Output produced | `Business_Operation_ml_ready.csv` |

